In [2]:
!pip install anndata==0.8.0

In [1]:
from samap.mapping import SAMAP
from samap.analysis import (get_mapping_scores, GenePairFinder, transfer_annotations,
                            sankey_plot, chord_plot, CellTypeTriangles, 
                            ParalogSubstitutions, FunctionalEnrichment,
                            convert_eggnog_to_homologs, GeneTriangles)
from samalg import SAM
import pandas as pd
from Bio import SeqIO
from samap.utils import (save_samap, load_samap)
import scanpy as sc
import matplotlib.colors
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats
from scipy import sparse 
from scipy import cluster
import seaborn as sns
import random
import sklearn
from sklearn.neighbors import KernelDensity
import time
import dill
import pickle
from scipy.spatial import distance
import os
import scipy

In [2]:
with open('../../parent_dict.pkl', 'rb') as f:
    parent_dict = pickle.load(f)

In [195]:
fn = '../../Active_SAM_joined/SAM_AC_ncbi_soupx_cleaned_03122025.h5ad'

In [196]:
sam = SAM()
sam.load_data(fn)

In [197]:
lsaturn_mv = pd.read_csv('../../SATURN_mapping/AC_vole_mouse_SATURN_30_seed.csv', index_col = 'Unnamed: 0')
lsaturn_m = pd.read_csv('../../SATURN_mapping/AC_30_seeds.csv', index_col = 'barcode')

In [198]:
ind = pd.read_csv('../../Active_SAMap_Joined/AC_MG_mapping_cleaned_03122025_0.csv')['Unnamed: 0']
lsamap_m = pd.DataFrame(index = list(ind))
for i in range(30):
    df = pd.read_csv('../../Active_SAMap_Joined/AC_MG_mapping_cleaned_03122025_'+str(i)+'.csv', index_col = 'Unnamed: 0')
    lsamap_m = pd.concat([lsamap_m, df], axis = 1)

In [199]:
lsamap_m

,0,1,2,3,4,5,6,7,8,9,...,20,21,22,23,24,25,26,27,28,29
AAACCCAGTTTGGAAA,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,...,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN
AAACGAAAGGACTGGT,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,...,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut,239 MARN-GRN Pyy Glut
AAACGAACAAGAGATT,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,...,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba,213 SCsg Gabrr2 Gaba
AAACGAACATGACGTT,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,...,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN
AAACGCTTCCTAGCGG,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,...,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN,327 Oligo NN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTGTTGTCATATGGC,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,...,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba,207 SCs Dmbx1 Gaba
TTTGTTGTCCAAACCA,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,022 L5 ET CTX Glut,023 SUB-ProS Glut,...,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,022 L5 ET CTX Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut,014 LA-BLA-BMA-PA Glut
TTTGTTGTCGCAGTCG,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,...,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba
TTTGTTGTCTACTCAT,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,...,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut,175 SC Bnc2 Glut


In [201]:
meta_folder = '../../Active_SAMap_Joined/AC_metadata/'

In [202]:
mn = os.listdir('../../Active_SAMap_Joined/AC_metadata/')

In [203]:
pd.read_csv(meta_folder + mn[3])

,Unnamed: 0,orig.ident,nCount_RNA,nFeature_RNA,n_counts,n_genes,key,eq_subclass,eq_subclass_lc,eq_subclass_frac,...,neurotransmitter,eq_subclass_nounlabeled_NN,eq_subclass_nounlabeled_nmm,nCount_SCT,nFeature_SCT,SCT_snn_res.0.8,seurat_clusters,SCT_snn_res.5,subclass_id_label_mapping,subclass_id_label_lc
0,AAACCCAGTTTGGAAA,AC,2108.0,1113,2108.0,1113,Run12_sample1,327 Oligo NN,44,1.0,...,unclear,327 Oligo NN,327 Oligo NN,2807.0,1105,2,10,10,327 Oligo NN,32
1,AAACGAAAGGACTGGT,AC,2050.0,1078,2050.0,1078,Run12_sample1,Unlabeled,164,1.0,...,Gaba,ac_35,ac_35,2832.0,1071,21,33,33,Unlabeled,167
2,AAACGAACAAGAGATT,AC,4400.0,2075,4400.0,2075,Run12_sample1,Unlabeled,5,1.0,...,Gaba,ac_9,ac_9,3810.0,2050,20,31,31,Unlabeled,5
3,AAACGAACATGACGTT,AC,2572.0,1336,2572.0,1336,Run12_sample1,327 Oligo NN,12,1.0,...,unclear,327 Oligo NN,327 Oligo NN,2905.0,1320,2,30,30,327 Oligo NN,267
4,AAACGCTTCCTAGCGG,AC,7505.0,2390,7505.0,2390,Run12_sample1,327 Oligo NN,137,1.0,...,unclear,327 Oligo NN,327 Oligo NN,4082.0,1961,33,66,66,327 Oligo NN,139
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
47895,TTTGTTGTCATATGGC,AC,2901.0,1549,2901.0,1549,Run17_sample4,Unlabeled,268,1.0,...,Gaba,ac_24,cj_ac_7,3488.0,1546,6,39,39,Unlabeled,254
47896,TTTGTTGTCCAAACCA,AC,8108.0,3072,8108.0,3072,Run17_sample4,Unlabeled,113,1.0,...,Glut,ac_29,ac_29,4892.0,2746,17,79,79,Unlabeled,117
47897,TTTGTTGTCGCAGTCG,AC,2290.0,1251,2290.0,1251,Run17_sample4,106 PVpo-VMPO-MPN Hmx2 Gaba,83,1.0,...,Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,106 PVpo-VMPO-MPN Hmx2 Gaba,3408.0,1251,0,50,50,106 PVpo-VMPO-MPN Hmx2 Gaba,88
47898,TTTGTTGTCTACTCAT,AC,2500.0,1452,2500.0,1452,Run17_sample4,Unlabeled,100,1.0,...,Glut,ac_16,ac_16,3395.0,1448,1,18,18,Unlabeled,126


In [204]:
test = pd.read_csv(meta_folder + mn[0])['subclass_id_label_mapping']

In [205]:
barcodes = pd.read_csv(meta_folder + mn[0])['Unnamed: 0']

In [206]:
raw_lc = pd.read_csv(meta_folder + mn[0])['subclass_id_label_lc']

In [207]:
df_lc = pd.DataFrame(data = list(raw_lc), index = list(barcodes), columns = ['raw_lc'])

In [208]:
mode_df = pd.DataFrame(index = [a for a in range(len(test))], columns = [b for b in range(len(mn))])
for j in range(len(mn)):
    print(mn[j])
    dat = list(pd.read_csv(meta_folder + mn[j])['subclass_id_label_lc'])
    mode_df.loc[:,j] = dat

AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_12.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_27.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_19.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_29.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_22.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_10.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_5.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_14.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_7.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_17.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_8.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_24.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_21.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_2.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_16.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_6.csv
AC_ncbi_soupx_metadata_subclass_250_cleaned_0

In [209]:
fin = []
for item in mode_df.columns:
    fin.append(mode_df.loc[10000,item])

In [210]:
scipy.stats.mode(fin)

ModeResult(mode=array([243]), count=array([30]))

In [211]:
import re

mn = sorted(mn, key=lambda s: [int(t) if t.isdigit() else t.lower() for t in re.split(r'(\d+)', s)])

In [212]:
mn

['AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_0.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_1.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_2.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_3.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_4.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_5.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_6.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_7.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_8.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_9.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_10.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_11.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_12.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_13.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_03122025_14.csv',
 'AC_ncbi_soupx_metadata_subclass_250_cleaned_0312

In [213]:
lsamap_mv = pd.DataFrame(index = [a for a in barcodes], columns = [b for b in range(len(mn))])
for j in range(len(mn)):
    dat = list(pd.read_csv(meta_folder + mn[j])['subclass_id_label_mapping'])
    lsamap_mv.loc[:,j] = dat

In [20]:
#lsamap_m = lsamap_mv

In [21]:
#for quail
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] == '067 LSX Sall3 Pax6 Gaba':
            inp = ['cj_m066_m067']
        if inp[0] == '136 PMv-TMv Pitx2 Glut':
            inp = ['cj_m136_m138']
        if inp[0] == '099 SBPV-PVa Six6 Satb2 Gaba':
            inp = ['cj_m091_m099']
        b += len(barcodes)
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [27]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [28]:
b/len(sam.adata)

0.013791166162818695

In [29]:
a/len(sam.adata)

0.644546403060542

In [30]:
len(samap_barcodes)/len(sam.adata)

0.1832312260895155

In [31]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.12872200968458225

In [40]:
(len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))

0.4126323914068865

In [32]:
len(saturn_barcodes)/len(sam.adata)

0.0066882474116482515

In [33]:
subset = sam.adata[sam.adata.obs_names.isin(samap_barcodes)]

In [34]:
a = 0 
for item in subset.obs['ss_subclass'].unique():
    if parent_dict[parent_dict[item]] == 'hypo':
        if len(subset[subset.obs['ss_subclass'] == item]) == len(sam.adata.obs[sam.adata.obs['ss_subclass'] == item]):
            a += 1
            print(item)

076 MEA-BST Lhx6 Nfib Gaba
135 STN-PSTN Pitx2 Glut
107 DMH Hmx2 Gaba
097 PVHd-SBPV Six3 Prox1 Gaba
134 PH-ant-LHA Otp Bsx Glut
104 TU-ARH Otp Six6 Gaba


In [35]:
a

6

In [214]:
#for anole
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] == '086 MPO-ADP Lhx8 Gaba':
            inp = ['ac_m058_m086']
        if inp[0] == '124 MPN-MPO-PVpo Hmx2 Glut':
            inp = ['ac_m124_m130']
        if inp[0] == '136 PMv-TMv Pitx2 Glut':
            inp = ['ac_m136_m138']
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [215]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [216]:
b/len(sam.adata)

0.02035490605427975

In [217]:
a/len(sam.adata)

0.6907306889352819

In [218]:
len(samap_barcodes)/len(sam.adata)

0.13872651356993737

In [219]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.1363465553235908

In [220]:
(len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))

0.4956739526411657

In [221]:
len(saturn_barcodes)/len(sam.adata)

0.013841336116910229

In [222]:
subset = sam.adata[sam.adata.obs_names.isin(samap_barcodes)]

In [223]:
a = 0 
for item in subset.obs['ss_subclass'].unique():
    if parent_dict[parent_dict[item]] == 'hypo':
        if len(subset[subset.obs['ss_subclass'] == item]) == len(sam.adata.obs[sam.adata.obs['ss_subclass'] == item]):
            a += 1
            print(item)

098 AHN-SBPV-PVHd Pdrm12 Gaba
099 SBPV-PVa Six6 Satb2 Gaba
122 LHA-MEA Otp Glut
117 LHA Barhl2 Glut
118 ADP-MPO Trp73 Glut
073 MEA-BST Sox6 Gaba


In [224]:
a

6

In [178]:
#for xenopus
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[0] != inp[1] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        if inp[0] == '104 TU-ARH Otp Six6 Gaba':
            inp = ['xt_m098_m104']
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [179]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [180]:
b/len(sam.adata)

0.0018947916913384334

In [181]:
a/len(sam.adata)

0.582151062267592

In [182]:
len(samap_barcodes)/len(sam.adata)

0.2266644560763601

In [183]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.1718339215082542

In [184]:
(len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))

0.4312035661218425

In [185]:
len(saturn_barcodes)/len(sam.adata)

0.011795078278581748

In [186]:
subset = sam.adata[sam.adata.obs_names.isin(samap_barcodes)]

In [187]:
a = 0 
for item in subset.obs['ss_subclass'].unique():
    if parent_dict[parent_dict[item]] == 'hypo':
        if len(subset[subset.obs['ss_subclass'] == item]) == len(sam.adata.obs[sam.adata.obs['ss_subclass'] == item]):
            a += 1
            print(item)

088 BST Tac2 Gaba
107 DMH Hmx2 Gaba
074 MEA-BST Lhx6 Sp9 Gaba
090 BST-MPN Six3 Nrgn Gaba
136 PMv-TMv Pitx2 Glut
134 PH-ant-LHA Otp Bsx Glut
105 TMd-DMH Foxd2 Gaba
073 MEA-BST Sox6 Gaba
143 MM-ant Foxb1 Glut
091 ARH-PVi Six6 Dopa-Gaba


In [188]:
a

10

In [95]:
#for zebrafish
threshold = 0
a = 0
b = 0
saturn_barcodes = []
saturn_barcodes_samap_mouse = []
samap_barcodes = []
samap_barcodes_saturn_mouse = []
mapping_dict ={}
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    inp = ['Unlabeled', 'Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_mv = lsaturn_mv[lsaturn_mv.index.isin(barcodes)].values.ravel()
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_mv = lsamap_mv[lsamap_mv.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_mv,f_saturn_mv = scipy.stats.mode(ct_saturn_mv)
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_mv,f_samap_mv = scipy.stats.mode(ct_samap_mv)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    inp[0] = mode_saturn_mv[0]
    inp[1] = mode_samap_mv[0]
    if inp[0] == inp[1]:
        a += len(barcodes)
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] != inp[1]:
        samap_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[0] == 'Unlabeled' and mode_saturn_m[0] == inp[1]:
        samap_barcodes_saturn_mouse.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] !=inp[0]:
        saturn_barcodes.extend(list(barcodes))
    if inp[0] != inp[1] and inp[1] == 'Unlabeled' and mode_samap_m[0] ==inp[0]:
        saturn_barcodes_samap_mouse.extend(list(barcodes))
    if mode_samap_m[0] == mode_saturn_m[0]:
        inp[2] = mode_samap_m[0]
    if inp[1] != inp[2] and inp[1] != 'Unlabeled' and inp[2] != 'Unlabeled':
        print(inp[0],inp[1],inp[2],lc)
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) >1:
        print(inp)
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [101]:
(a + b + len(samap_barcodes) + len(saturn_barcodes) + len(saturn_barcodes_samap_mouse) + len(samap_barcodes_saturn_mouse))/len(sam.adata)

1.0

In [102]:
b/len(sam.adata)

0.0

In [103]:
a/len(sam.adata)

0.9239622334364317

In [104]:
len(samap_barcodes)/len(sam.adata)

0.03716425199413967

In [105]:
len(samap_barcodes_saturn_mouse)/len(sam.adata)

0.03887351456942862

In [106]:
(len(samap_barcodes_saturn_mouse)/len(sam.adata))/(len(samap_barcodes)/len(sam.adata) + len(samap_barcodes_saturn_mouse)/len(sam.adata))

0.5112395632626847

In [107]:
len(saturn_barcodes)/len(sam.adata)

0.0

In [108]:
subset = sam.adata[sam.adata.obs_names.isin(samap_barcodes)]

In [109]:
a = 0 
for item in subset.obs['ss_subclass'].unique():
    if parent_dict[parent_dict[item]] == 'hypo':
        if len(subset[subset.obs['ss_subclass'] == item]) == len(sam.adata.obs[sam.adata.obs['ss_subclass'] == item]):
            a += 1
            print(item)

086 MPO-ADP Lhx8 Gaba
105 TMd-DMH Foxd2 Gaba
091 ARH-PVi Six6 Dopa-Gaba
092 TMv-PMv Tbx3 Hist-Gaba


In [110]:
a

4

In [141]:
#for vole
threshold = .75
mapping_dict ={}
a = 0
b = 0
for lc in sam.adata.obs['eq_subclass_lc'].unique():
    a += 1
    inp = ['Unlabeled', 'Unlabeled']
    barcodes = sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc]
    ct_saturn_m = lsaturn_m[lsaturn_m.index.isin(barcodes)].values.ravel()
    ct_samap_m = lsamap_m[lsamap_m.index.isin(barcodes)].values.ravel()
    
    mode_saturn_m,f_saturn_m = scipy.stats.mode(ct_saturn_m)
    mode_samap_m,f_samap_m = scipy.stats.mode(ct_samap_m)
    
    if  f_saturn_m[0]/len(ct_saturn_m)> threshold:
        inp[0] = mode_saturn_m[0]
    if f_samap_m[0]/len(ct_samap_m) > threshold:
        inp[1] = mode_samap_m[0]
    if inp[1] != inp[0] and inp[0] != 'Unlabeled' and inp[1] != 'Unlabeled':
        b += len(sam.adata.obs_names[sam.adata.obs['eq_subclass_lc'] == lc])
        if inp[0] == '077 CEA-BST Gal Avp Gaba':
            mapping_dict[lc] = 'mg_077_106'
        elif inp[0] == '105 TMd-DMH Foxd2 Gaba':
            mapping_dict[lc] = 'mg_105_107'
    inp_val = list(set(inp) - set(['Unlabeled']))
    if len(inp_val) == 0:
        mapping_dict[lc] = 'Unlabeled'
    elif len(inp_val) == 1:
        mapping_dict[lc] = inp_val[0]

In [225]:
new_mapping = []
new_mapping_name = 'test'
for item in sam.adata.obs['eq_subclass_lc']:
    new_mapping.append(mapping_dict[item])
sam.adata.obs[new_mapping_name] = new_mapping

In [226]:
a = 0
fin_obs = []
for item in sam.adata.obs_names:
    if sam.adata.obs.loc[item,'test'] == sam.adata.obs.loc[item,'ss_subclass']:
        a += 1
        fin_obs.append(item)

In [39]:
sam.save_anndata(fn)